In [12]:
from pathlib import Path
import re
from scipy.stats import pearsonr, spearmanr

MODEL = "georsclip"
BACKBONE = f"{MODEL}-14"
ROOT = Path("/home/gridsan/manderson/ovdsat/run/eval/detection")

# RPN per-class recall
recall = {
    "dior": {
        "ship": .8515, "groundtrackfield": .8375, "baseballfield": .8243,
        "overpass": .3586, "expresswaytollstation": .1610, "vehicle": .6609,
        "basketballcourt": .8636, "tenniscourt": .8446, "airplane": .8424,
        "airport": .2115, "expresswayservicearea": .2569, "bridge": .4820,
        "trainstation": .0197, "storagetank": .7734, "stadium": .7881,
        "windmill": .5118, "harbor": .7455, "golffield": .0926,
        "chimney": .6840, "dam": .5055,
    },
    "nwpu": {
        "airplane": .9046, "ship": .9342, "storagetank": .7382,
        "baseballdiamond": .8798, "tenniscourt": .7870,
        "basketballcourt": .8356, "groundtrackfield": .9238,
        "harbor": .3651, "bridge": .6531, "vehicle": .8951,
    },
    "simd": {
        "car": .9910, "truck": .9735, "airliner": .9723,
        "stairtruck": .4172, "van": .9941, "bus": .9603,
        "longvehicle": .9781, "boat": .9966, "propeller": .9860,
        "chartered": .9626, "pushbacktruck": .7143, "other": .9556,
        "fighter": 1.0000, "trainer": .9871, "helicopter": .9463,
    },
    "mar": {
        "C17": .8956, "KC135": .9271, "B1B": .9020,
        "B52": .8835, "F16": .8921, "SU24": .7653,
        "TU95": .9439, "TU22": .8972, "F22": .8032,
        "F15": .8804, "C130": .9494, "E8": .9654,
        "E3": .9781, "C5": .9309, "KC1": .9242,
        "FA18": .9876, "P3C": .9748, "SU35": .8956,
        "SU34": .8831, "TU160": .9234,
    },
}

# Experiment directories
suffixes = {
    "dior": "new",
    "nwpu": "1",
    "simd": "2",
    "mar": "2",
}

def normalize(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

print(f"Testing {BACKBONE}\n")

for dataset, rpn in recall.items():
    rpn = {normalize(k): v for k, v in rpn.items()}
    scores = {c: {} for c in rpn}

    folder = (
        ROOT / dataset / "coop_prototypes"
        / f"backbone_{BACKBONE}_boxes"
        / f"{dataset}_{MODEL}_{suffixes[dataset]}"
        / "nctx4_cscFalse_ctpend"
    )

    if dataset == "dior":
        folder /= "0.5"

    # Read 10-shot results for splits 1, 2, 3
    missing_files = False

    for split in (1, 2, 3):
        file = folder / f"N10-{split}" / f"results_{BACKBONE}.txt"

        if not file.is_file():
            print(f"Missing: {file}")
            missing_files = True
            continue

        for line in file.read_text().splitlines():
            parts = line.split()
            if len(parts) < 7:
                continue

            cls = normalize(" ".join(parts[:-6]))
            if cls not in rpn:
                continue

            try:
                scores[cls][split] = (
                    float(parts[-2]),  # mAP50
                    float(parts[-1])   # mAP50-95
                )
            except ValueError:
                continue

    if missing_files:
        print(f"{dataset.upper()}: Skipping (missing result files)\n")
        continue

    # Average AP across 3 splits for each class
    # if dataset == "simd":
    #     print("Unmatched classes:",
    #           [c for c in rpn if len(scores[c]) != 3])
    classes = [c for c in rpn if len(scores[c]) == 3]

    print(f"{dataset.upper()} ({len(classes)}/{len(rpn)} classes)")

    if len(classes) < 3:
        print("Not enough classes for correlation\n")
        continue

    x = [rpn[c] for c in classes]

    for i, metric in enumerate(["mAP50"]):
        y = [
            sum(scores[c][split][i] for split in (1, 2, 3)) / 3
            for c in classes
        ]

        if len(set(x)) < 2 or len(set(y)) < 2:
            print(f"{metric}: Correlation undefined")
            continue

        r, p = pearsonr(x, y)
        rho, ps = spearmanr(x, y)

        print(
            #f"{metric}: Pearson r={r:.2e} (p={p:.2e}), "
            f"Spearman rho={rho:.2e} (p={ps:.2e})"
        )

    print()

Testing georsclip-14

DIOR (20/20 classes)
Spearman rho=8.97e-01 (p=8.59e-08)

NWPU (10/10 classes)
Spearman rho=5.52e-01 (p=9.84e-02)

SIMD (7/15 classes)
Spearman rho=2.14e-01 (p=6.45e-01)

MAR (20/20 classes)
Spearman rho=6.75e-01 (p=1.10e-03)

